# Organizational Transformation Analysis - Data Preparation Documentation

## Project Overview

This document outlines the data preparation process for an organizational health assessment conducted for a consulting firm's client. The analysis transformed raw employee survey data covering multiple organizational health dimensions into a structured analytical framework supporting strategic transformation decisions.

## 1. Data Sources & Initial State

### Input Data:

- **Survey responses:** 50+ questions across organizational health themes
- **Response format:** Mix of quantitative (Likert scale) and qualitative (open-text)
- **Demographics:** Department/Function, Seniority, Tenure
- **Format:** Wide-format table (one row per respondent, one column per question)

#### Key Challenges:

- Disparate questions without clear organizational structure
- Mixed response types (quantitative and qualitative) requiring different analytical approaches
- Need to map individual questions to broader strategic themes for executive reporting
- Inconsistent department naming requiring standardization

## 2. Data Cleaning Process

### 2.1 Initial Cleaning Steps

#### Removed unnecessary columns:

- Dropped erroneous "Roles and Responsibility" question column
- Removed timestamp column (not relevant for analysis)

#### Standardized categorical variables:

- Cleaned and standardized Department/Function values to ensure consistent grouping

## 3. Question Metadata Architecture

### 3.1 Conceptual Framework
To enable scalable analysis and clear reporting, survey questions were mapped to a hierarchical structure:

Theme (Survey Section)

    ↓
    
Driver (Specific aspect being measured)

    ↓
    
Question (Individual survey item)

**Example:**

- **Theme:** Structural Effectiveness
- **Driver:** Role Clarity
- **Question:** "I understand what is expected of me in my role"

### 3.2 Question Metadata Table

Created a reference table mapping each question to its analytical context:

| Column | Purpose | Example Values |
| :--- | :--- | :--- |
| **question_id** | Short identifier for joins and debugging | Q1, Q2, Q3... |
| **question_text** | Full question text for traceability | "I understand what is expected..." |
| **theme** | Organizational health dimension | Structural Effectiveness, Culture, Governance |
| **driver** | Specific sub-topic being measured | Role Clarity, Decision Speed, Leadership Role-modeling |
| **response_type** | Response format | Quantitative, Qualitative |
| **driver_type** | Change lever classification | Behavioral, Operational |

#### Driver Type Classification Logic:

- Behavioral: Improvement requires changes in how people behave (e.g., leadership role-modeling)
- Operational: Improvement requires structural/process/system changes (e.g., decision-making workflows)

### 3.3 Benefits of This Architecture

#### Analytical advantages:

- Map questions → themes → drivers once, reuse logic everywhere
- Enable clean aggregation at theme and driver levels
- Support filtering and drill-down capabilities

#### Reporting clarity:

- Explain results meaningfully: "Structural Effectiveness is low mainly due to decision clarity and process efficiency"

## 4. Data Transformation

### 4.1 Wide-to-Long Format Conversion

#### Rationale:

- Enables joining with question metadata table
- Supports flexible aggregation and filtering
- Simplifies DAX measure logic in Power BI

#### Original structure (wide):

| response_id | Department | Q1 | Q2 | Q3 | ... |
|-------------|------------|----|----|----|----|
| R001        | Finance    | 4  | 3  | 5  | ... |

#### Transformed structure (long):

| response_id | Department | question_id | response       |
|-------------|------------|-------------|----------------|
| R001        | Finance    | Q1          | 4              |
| R001        | Finance    | Q2          | 3              |
| R001        | Finance    | Q3          | 5              |

#### Added fields:

- **response_id:** Unique identifier per respondent
- **question_id:** Standardized question reference

## 5. Quantitative Analysis Setup

### 5.1 Scoring Model - DAX Measures

#### Driver-Level Score:
```dax
Driver Score :=
CALCULATE(
    AVERAGE(responses_long[response_numeric]),
    question_metadata[response_type] = "Quantitative"
)
```
Calculates average Likert score for all questions under a driver.

#### Theme-Level Score:

```dax
daxTheme Score :=
AVERAGEX(
    VALUES(question_metadata[driver]),
    [Driver Score]
)
```
Aggregates driver scores to theme level, ensuring equal weighting across drivers.

## 6. Qualitative Data Processing

### Why NLP Preprocessing for Qualitative?

- **Consistency:** Standardized text enables pattern detection
- **Quality**: Flagging low-information responses improves analysis focus
- **Comparability:** Lemmatization groups word variants (e.g., "lead," "leading," "leadership")

### 6.1 Data Preparation

#### Filtering and setup:
```python
# Merge responses with question metadata
df_merged = response_df.merge(
    question_df,
    on="Question_id",
    how="left"
)

# Filter to qualitative responses only
df_qual = df_merged[df_merged["Response_type"] == "Qualitative"].copy()
```

### 6.2 Text Cleaning Pipeline

#### Step 1: Basic cleaning

```python
def basic_clean(text):
    if pd.isna(text):
        return ""
    text = text.lower()                    # Lowercase
    text = re.sub(r"[^\w\s]", " ", text)  # Remove punctuation
    text = re.sub(r"\s+", " ", text)      # Normalize whitespace
    return text.strip()

df_qual["clean_text"] = df_qual["Response"].apply(basic_clean)
```

#### Step 2: Stopword removal
```python
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    return " ".join(
        word for word in text.split()
        if word not in stop_words
    )

df_qual["clean_text_stop"] = df_qual["clean_text"].apply(remove_stopwords)
```
#### Step 3: Lemmatization
```python
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatise(text):
    return " ".join(
        lemmatizer.lemmatize(word)
        for word in text.split()
    )

df_qual["clean_text_lemma"] = df_qual["clean_text_stop"].apply(lemmatise)
```
#### Step 4: Data Quality Flagging

Low-information response detection:

```python
def low_information(text):
    if not text:
        return True
    if text in ["na", "n a", "none", "nil"]:
        return True
    if len(text.split()) < 2:
        return True
    return False

df_qual["low_information_flag"] = df_qual["clean_text"].apply(low_information)
```
**Purpose:**
Flags non-substantive responses (e.g., "N/A", single words) for exclusion from text analysis while retaining for completeness tracking.

## 7. Data Outputs

### responses_long.csv
- Long-format response table
- Joined with question metadata
- Ready for Power BI import

### Question Metadata.csv
- Reference table for all questions
- Theme and driver mappings
- Response type classifications

#### Enables:
- Filtering by theme, driver, department, seniority
- Drill-down from theme → driver → question level
- Integrated quantitative and qualitative analysis